# 04 — GRU từ paper BAF

Nguồn: *Objective over Architecture*, DOI10.3390/computation13120290; repo commit `6e8252cb9080378905b2aae30033c5946b72335b`. Model và training cùng nằm trong `src/gru.py`. Kiến trúc và tham số fixed recipe được giữ; code được viết lại cho dễ đọc.

`author_fixed`: random64/16/20, giữ preprocessing outer80% như code tác giả, có feature của inner validation trong preprocessing. Random split gồm mọi tháng, kể cả7. Không chạy lại HPO10 trial.

`temporal`: chuyển sang train0–5/validation6 với schema của nhóm, train-only preprocessing; không dự đoán7. Đây là adaptation.

GRU xử lý chuỗi các cột trong từng hồ sơ, không phải lịch sử giao dịch theo thời gian. Weighted loss là cấu hình nguồn; unweighted là ablation thêm.

## 1. Môi trường
Chọn kernel có PyTorch phù hợp GPU. Dùng môi trường riêng cho GRU; xem README. Không tự cài CUDA hoặc đổi cấu hình khi OOM.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = None  # Ví dụ: r"D:\UIT\DS317\BAF_project_clean"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (path for path in [here, *here.parents] if (path / "src/baf.py").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở project BAF_project_clean hoặc điền PROJECT_DIR.")

sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf

if Path(baf.__file__).resolve() != (ROOT / "src/baf.py").resolve():
    raise RuntimeError(
        "Kernel đang giữ module của project khác. Restart Kernel trước khi Run All."
    )

print("Project:", ROOT)
print("Kernel:", sys.executable)
print(pd.Series(baf.package_versions()).to_string())

try:
    import gru
except ImportError as error:
    raise RuntimeError(
        "Cài PyTorch theo https://pytorch.org/get-started/locally/ và chọn đúng kernel GRU."
    ) from error

print("Torch:", gru.torch.__version__)
print("CUDA:", gru.torch.cuda.is_available())
print(
    "GPU:",
    gru.torch.cuda.get_device_name(0) if gru.torch.cuda.is_available() else "CPU",
)


Project: D:\UIT\DS317\BAF_project
Kernel: c:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe
numpy            2.4.2
pandas           3.0.1
scipy           1.17.1
scikit-learn     1.7.2
psutil           7.2.2
lightgbm         4.7.0
Torch: 2.14.1+cpu
CUDA: False
GPU: CPU


## 2. Cấu hình
Mặc định pilot8192 train, tối đa4096 mỗi tập đánh giá, 1 epoch. Chưa dùng điểm pilot cho benchmark. Full recipe dùng embedding186, hidden733, Adam lr0.000448103301026688, batch256, dropout0.3, 8 epoch. Giữ bản notebook có output cho từng giai đoạn pilot/full.

In [2]:
DATA_PATH = ROOT / "data/raw/Base.csv"
previous_data = ROOT.parent / "BAF_project/data/raw/Base.csv"
if not DATA_PATH.is_file() and previous_data.is_file():
    DATA_PATH = previous_data

EXPECTED_ROWS = 1_000_000
CONFIG = gru.Config(
    protocol="author_fixed",
    mode="pilot",
    seeds=[1234],
    threads=4,
    device="auto",
    embedding_dim=186,
    hidden_dim=733,
    learning_rate=0.000448103301026688,
    batch_size=256,
    epochs=8,
    dropout=0.3,
)

RUN_DIR = baf.new_directory(
    ROOT / "results/runs",
    f"gru_{CONFIG.protocol}_{CONFIG.mode}",
)
print(CONFIG)
print("Output:", RUN_DIR)


Config(protocol='author_fixed', mode='pilot', methods=['none', 'balanced'], seeds=[1234], threads=4, device='auto', embedding_dim=186, hidden_dim=733, learning_rate=0.000448103301026688, epochs=8, batch_size=256, dropout=0.3, pilot_train=8192, pilot_eval=4096, save_models=True, dataset_tag='BAF_Base')
Output: D:\UIT\DS317\BAF_project\results\runs\gru_author_fixed_pilot_20261027_101932_1668bf5a


## 3. Huấn luyện hai loss
Cùng khởi tạo và minibatch order theo seed. Cấu hình fixed không đảm bảo tối ưu cho cả hai loss. Peak RSS là tiến trình; peak CUDA allocation được ghi riêng.

In [3]:
metrics = gru.train(
    data_path=DATA_PATH,
    output=RUN_DIR,
    config=CONFIG,
    expected_rows=EXPECTED_ROWS,
)
display(metrics)


D:\UIT\DS317\BAF_project\src\gru.py:292: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(part.fraud_bool.to_numpy(dtype=np.int64)),


{
  "protocol": "author_fixed",
  "mode": "pilot",
  "methods": [
    "none",
    "balanced"
  ],
  "seeds": [
    1234
  ],
  "threads": 4,
  "device": "auto",
  "embedding_dim": 186,
  "hidden_dim": 733,
  "learning_rate": 0.000448103301026688,
  "epochs": 8,
  "batch_size": 256,
  "dropout": 0.3,
  "pilot_train": 8192,
  "pilot_eval": 4096,
  "save_models": true,
  "dataset_tag": "BAF_Base",
  "evaluation_partitions": [
    "validation",
    "random_test"
  ]
}
                n positive                    months
train        8192       90  [0, 1, 2, 3, 4, 5, 6, 7]
validation   4096       45  [0, 1, 2, 3, 4, 5, 6, 7]
random_test  4096       45  [0, 1, 2, 3, 4, 5, 6, 7]
Author fixed recipe: random64/16/20, preprocessing fit outer80% as code. Inner validation features enter preprocessing. No HPO. Seed reset and paired unweighted ablation added. Random split includes month7. PILOT subsample/one epoch: not a full BAF benchmark.
EPOCH {"method": "none", "seed": 1234, "epoch": 1, "loss_ba

,protocol,mode,method,seed,partition,threshold,fit_seconds,peak_rss_mib,peak_cuda_allocated_mib,initial_model_sha256,...,precision_frozen_threshold,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp
0,author_fixed,pilot,gru_none,1234,validation,0.026116,50.052454,680.886719,None,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.086735,0.047852,0.000000,0.000000,0.983551,0.989014,3872,179,28,17
1,author_fixed,pilot,gru_none,1234,random_test,0.026116,50.052454,680.886719,None,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.049751,0.049072,0.000000,0.000000,0.983551,0.989014,3860,191,35,10
2,author_fixed,pilot,gru_balanced,1234,validation,0.775013,48.323012,673.820312,None,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.103448,0.042480,0.622222,0.025854,0.839471,0.738281,3895,156,27,18
3,author_fixed,pilot,gru_balanced,1234,random_test,0.775013,48.323012,673.820312,None,e00a83b98708a70d9412a23dbb7313461a79f986166f1e...,...,0.074257,0.049316,0.600000,0.023417,0.827737,0.720703,3864,187,30,15


## 4. Kiểm chứng kết quả
Recall tại0.5 gần cách đánh giá paper hơn; không so trực tiếp số0.78 của paper với Recall@FPR5% ở notebook03. Pilot chỉ dùng đo tài nguyên.

In [4]:
report = baf.audit_results([RUN_DIR], reference="gru_none")
display(report.summary)
display(report.deltas)
display(pd.read_csv(RUN_DIR / "epoch_history.csv"))


,protocol,mode,method,partition,n_runs,ap_mean,ap_std,recall_mean,recall_std,fit_seconds_mean
0,author_fixed,pilot,gru_balanced,random_test,1,0.086734,NaN,0.333333,NaN,48.323012
1,author_fixed,pilot,gru_balanced,validation,1,0.129664,NaN,0.400000,NaN,48.323012
2,author_fixed,pilot,gru_none,random_test,1,0.035636,NaN,0.222222,NaN,50.052454
3,author_fixed,pilot,gru_none,validation,1,0.054208,NaN,0.377778,NaN,50.052454


,reference,method,seed,partition,delta_recall_pp,delta_ap
0,gru_none,gru_balanced,1234,validation,2.222222,0.075456
1,gru_none,gru_balanced,1234,random_test,11.111111,0.051098


,method,seed,epoch,loss_batch_mean,seconds
0,none,1234,1,0.096122,50.033405
1,balanced,1234,1,0.648265,48.310880


## 5. Xuất proof trong một lần
Dùng cùng hàm với các notebook khác. Code, log huấn luyện, epoch history, preprocessing, checkpoint và predictions được giữ trong ZIP. Notebook kiểm chứng được tạo từ output thực tế đã lưu, không phải bản sao giao diện chưa lưu.

In [5]:
archive = baf.export_proof(
    run_paths=[RUN_DIR],
    output_root=ROOT / "results/proofs",
    reference="gru_none",
)


Verified prediction files: 4
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_102208_7abf13ba.zip
